In [1]:
from neo4j import GraphDatabase
import pandas as pd
import os

# Reconnect to the database
URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "12345678") # Make sure your password is correct

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Successfully reconnected to Neo4j!")

# Redefine the helper function just to be safe
def run_query(query, parameters=None):
    with driver.session() as session:
        result = session.run(query, parameters)
        return pd.DataFrame([r.values() for r in result], columns=result.keys())

Successfully reconnected to Neo4j!


In [ ]:
# Define your true absolute base directory
BASE_DIR = r"D:\\EIDIA 5\\Knowledge\\VsCode_Knowledge\\euromed-graph-course\\miniproject2_yassir"

def execute_cypher_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as file:
        queries = file.read().split(';')
        for query in queries:
            if query.strip():
                try:
                    run_query(query)
                except Exception as e:
                    print(f"Error executing snippet: {e}")
    print(f"Successfully executed {filepath}")

# Construct the absolute paths for the Cypher scripts
constraints_file = os.path.join(BASE_DIR, 'cypher', '01_constraints_indexes.cypher')
load_data_file = os.path.join(BASE_DIR, 'cypher', '02_load_data.cypher')

# Execute the constraints file
execute_cypher_file(constraints_file)

Error executing snippet: {code: Neo.ClientError.Schema.EquivalentSchemaRuleAlreadyExists} {message: An equivalent constraint already exists, 'Constraint( id=3, name='candidate_id_unique', type='NODE PROPERTY UNIQUENESS', schema=(:Candidate {id}), ownedIndex=2, graphTypeDependence='UNDESIGNATED' )'.}
Error executing snippet: {code: Neo.ClientError.Schema.EquivalentSchemaRuleAlreadyExists} {message: An equivalent constraint already exists, 'Constraint( id=5, name='skill_id_unique', type='NODE PROPERTY UNIQUENESS', schema=(:Skill {id}), ownedIndex=4, graphTypeDependence='UNDESIGNATED' )'.}
Error executing snippet: {code: Neo.ClientError.Schema.EquivalentSchemaRuleAlreadyExists} {message: An equivalent constraint already exists, 'Constraint( id=7, name='job_id_unique', type='NODE PROPERTY UNIQUENESS', schema=(:Job {id}), ownedIndex=6, graphTypeDependence='UNDESIGNATED' )'.}
Error executing snippet: {code: Neo.ClientError.Schema.EquivalentSchemaRuleAlreadyExists} {message: An equivalent con

In [3]:
import os
import shutil

# 1. Wipe the database to ensure a clean slate
def clear_database():
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
    print("Database wiped clean.")

clear_database()

# 2. Get the exact Import folder Neo4j is currently using
try:
    settings = run_query("SHOW SETTINGS YIELD name, value WHERE name = 'server.directories.import' RETURN value")
    neo4j_import_path = settings.iloc[0]['value']
    print(f"Neo4j's exact import folder is: {neo4j_import_path}")

    # 3. Automatically copy all CSVs from your VS Code project to the Neo4j import folder
    source_dir = os.path.join(BASE_DIR, 'data')
    for filename in os.listdir(source_dir):
        if filename.endswith('.csv'):
            source_file = os.path.join(source_dir, filename)
            dest_file = os.path.join(neo4j_import_path, filename)
            shutil.copy2(source_file, dest_file)
            
    print("Successfully copied all CSV files to the correct Neo4j import directory.")

except Exception as e:
    print(f"Warning during file copy: {e}")

# 4. Now execute the Cypher load script again!
execute_cypher_file(load_data_file)
print("Graph data populated successfully!")

Database wiped clean.
Neo4j's exact import folder is: C:\Users\fftt7\.Neo4jDesktop2\Data\dbmss\dbms-d617abf1-c98c-4207-a72a-00c2c87ed9f1\import
Successfully copied all CSV files to the correct Neo4j import directory.
Successfully executed D:\\EIDIA 5\\Knowledge\\VsCode_Knowledge\\euromed-graph-course\\session_4\\project2_yassir\cypher\02_load_data.cypher
Graph data populated successfully!


In [4]:
cq01_query = """
MATCH (c:Candidate {id: 'C01'})-[hs:HAS_SKILL]->(s:Skill)
RETURN c.name AS Candidate, s.canonical_label AS Skill, hs.level AS Level, hs.verification AS Verification_Status;
"""
df_cq01 = run_query(cq01_query)
df_cq01

,Candidate,Skill,Level,Verification_Status
0,Amina Idrissi,Python,3,Verified
1,Amina Idrissi,Cypher,2,Verified
2,Amina Idrissi,Neo4j,3,Verified
3,Amina Idrissi,Docker,1,Unverified


In [5]:
cq02_query = """
MATCH (j:Job {id: 'J01'})-[req:REQUIRE {mandatory: true}]->(req_skill:Skill)
WITH j, count(req_skill) AS total_mandatory
MATCH (c:Candidate)-[hs:HAS_SKILL]->(s:Skill)<-[req:REQUIRE {mandatory: true}]-(j)
WHERE hs.verification = 'Verified' AND hs.level >= req.minimum_level
WITH j, total_mandatory, c, count(s) AS candidate_matches
WHERE candidate_matches = total_mandatory
RETURN j.id AS JobID, c.id AS CandidateID, c.name AS CandidateName;
"""
df_cq02 = run_query(cq02_query)
df_cq02

,JobID,CandidateID,CandidateName
0,J01,C01,Amina Idrissi


In [6]:
cq08_query = """
MATCH (ae:ApplicationEvent)-[:APPLIED_TO]->(j:Job)
RETURN j.id AS JobID, j.title AS JobTitle, ae.event_type AS EventType, count(ae) AS EventCount
ORDER BY j.id, EventType;
"""
df_cq08 = run_query(cq08_query)
df_cq08

,JobID,JobTitle,EventType,EventCount
0,J01,Graph Data Engineer,Applied,3
1,J01,Graph Data Engineer,Reapplied,2
2,J01,Graph Data Engineer,Withdrawn,2
3,J02,ML Engineer,Applied,1
4,J03,Data Analyst,Applied,2
5,J05,Backend Developer,Applied,1
6,J07,Data Engineering Intern,Applied,1


### Testing section

In [7]:
# Test Assertions 1, 2, and 4
test_match_query = """
MATCH (j:Job)-[req:REQUIRE {mandatory: true}]->(req_skill:Skill)
WITH j, count(req_skill) AS total_mandatory
MATCH (c:Candidate)-[hs:HAS_SKILL]->(s:Skill)<-[req:REQUIRE {mandatory: true}]-(j)
WHERE hs.verification = 'Verified' AND hs.level >= req.minimum_level
WITH j, total_mandatory, c, count(s) AS candidate_matches
WHERE candidate_matches = total_mandatory
RETURN j.id AS JobID, collect(c.id) AS EligibleCandidates
ORDER BY JobID;
"""
df_test_match = run_query(test_match_query)
display(df_test_match)

,JobID,EligibleCandidates
0,J01,[C01]
1,J02,[C03]
2,J03,"[C02, C06, C08]"
3,J07,"[C01, C02, C03, C05, C07, C09, C10]"


In [8]:
# Test Assertion 3: Geography Rule
test_geo_query = """
MATCH (j:Job {id: 'J03'})-[req:REQUIRE {mandatory: true}]->(s:Skill)
WITH j, count(s) AS total_mandatory
MATCH (c:Candidate)-[hs:HAS_SKILL]->(s:Skill)<-[req:REQUIRE {mandatory: true}]-(j)
WHERE hs.verification = 'Verified' AND hs.level >= req.minimum_level
WITH j, total_mandatory, c, count(s) AS candidate_matches
WHERE candidate_matches = total_mandatory
MATCH (j)-[:LOCATED_AT]->(city:City)
MATCH (c)-[:LIVES_IN|AVAILABLE_ONSITE_IN]->(city)
RETURN j.id AS JobID, c.id AS CandidateID, city.name AS JobCity, c.remote_available AS IsRemote;
"""
df_test_geo = run_query(test_geo_query)
display(df_test_geo)

,JobID,CandidateID,JobCity,IsRemote
0,J03,C02,Casablanca,True
1,J03,C02,Casablanca,True
2,J03,C06,Casablanca,False
3,J03,C06,Casablanca,False


In [9]:
test_history_query = """
MATCH (c:Candidate {id: 'C02'})-[:MADE_ACTION]->(ae:ApplicationEvent)-[:APPLIED_TO]->(j:Job {id: 'J01'})
RETURN c.id AS Candidate, j.id AS Job, ae.event_type AS Action, ae.timestamp AS Timestamp
ORDER BY ae.timestamp ASC;
"""
df_test_history = run_query(test_history_query)
display(df_test_history)

,Candidate,Job,Action,Timestamp
0,C02,J01,Applied,2026-09-02T10:30:00.000000000+00:00
1,C02,J01,Withdrawn,2026-09-04T11:00:00.000000000+00:00
2,C02,J01,Reapplied,2026-09-06T08:00:00.000000000+00:00


In [10]:
test_alias_query = """
CALL {
    MATCH (a:Alias {name: 'ML'})-[:RESOLVES_TO]->(s:Skill)
    RETURN 'Alias Resolution' AS Test, a.name + ' -> ' + s.canonical_label AS Result
    UNION
    MATCH (s:Skill) WHERE s.canonical_label CONTAINS 'Java'
    RETURN 'Distinct Skills' AS Test, s.canonical_label AS Result
}
RETURN Test, collect(Result) AS Outputs;
"""
df_test_alias = run_query(test_alias_query)
display(df_test_alias)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }} {position: line: 2, column: 1, offset: 1} for query: "\nCALL {\n    MATCH (a:Alias {name: 'ML'})-[:RESOLVES_TO]->(s:Skill)\n    RETURN 'Alias Resolution' AS Test, a.name + ' -> ' + s.canonical_label AS Result\n    UNION\n    MATCH (s:Skill) WHERE s.canonical_label CONTAINS 'Java'\n    RETURN 'Distinct Skills' AS Test, s.canonical_label AS Result\n}\nRETURN Test, collect(Result) AS Outputs;\n"


,Test,Outputs
0,Alias Resolution,"[ML -> HTML, ML -> Machine Learning]"
1,Distinct Skills,"[Java, JavaScript]"


In [11]:
test_stretch_query = """
MATCH (c:Candidate)-[:EXPERIENCED]->(e:Experience)
MATCH (e)-[:LEARNED]->(s1:Skill {category: 'Programming'})
MATCH (e)-[:LEARNED]->(s2:Skill {category: 'Graph database'})
RETURN c.name AS Candidate, e.role AS Role, s1.canonical_label AS ProgrammingSkill, s2.canonical_label AS GraphSkill;
"""
df_test_stretch = run_query(test_stretch_query)
display(df_test_stretch)

,Candidate,Role,ProgrammingSkill,GraphSkill
0,Amina Idrissi,Graph Developer,Python,Neo4j


In [12]:
driver.close()
print("Connection closed.")

Connection closed.
